# Chapter 1: Profiling & Benchmarking

Before optimizing anything, we need to **measure** where our model spends time and memory.

This notebook walks through three profiling tools:
1. **benchmarking_script.py** — Python-level timing (timeit)
2. **nsys_profile.py** — NVIDIA Nsight Systems GPU kernel profiling
3. **memory_profiling.py** — PyTorch memory snapshots & peak tracking

---

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import timeit

from utils.model_loader import create_model, generate_random_batch
from utils.benchmarking import benchmark_fn, print_gpu_info, BenchmarkResult

print_gpu_info()

## Part 1: End-to-End Benchmarking

We create a transformer model from hyperparameters and time its forward pass.

In [ ]:
# Model hyperparameters
NUM_LAYERS = 6
HIDDEN_DIM = 1024
NUM_HEADS = 8
VOCAB_SIZE = 32000

# Data
BATCH_SIZE = 8
SEQ_LEN = 512

# Timing
WARMUP = 5
STEPS = 20

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Initialize model
model = create_model(
    num_layers=NUM_LAYERS,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    vocab_size=VOCAB_SIZE,
    device=device,
)
print(f"Parameters: {model.param_count_str()} ({model.param_count():,})")

# Generate random data
input_ids = generate_random_batch(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE, device)
print(f"Input shape: {input_ids.shape}")

In [ ]:
# Benchmark: Forward pass only
def forward_step():
    with torch.no_grad():
        _ = model(input_ids)

result_fwd = benchmark_fn(
    forward_step,
    warmup_steps=WARMUP,
    measure_steps=STEPS,
    name="Forward Pass",
)
print(result_fwd.summary())

In [ ]:
# Benchmark: Forward + Backward
targets = generate_random_batch(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE, device)
loss_fn = nn.CrossEntropyLoss()

def fwd_bwd_step():
    model.zero_grad(set_to_none=True)
    logits = model(input_ids)
    loss = loss_fn(logits.view(-1, logits.size(-1)), targets.view(-1))
    loss.backward()

result_both = benchmark_fn(
    fwd_bwd_step,
    warmup_steps=WARMUP,
    measure_steps=STEPS,
    name="Forward + Backward",
)
print(result_both.summary())

In [ ]:
# Compare forward vs forward+backward
print(f"\nForward only:      {result_fwd.mean_ms:.2f} ms")
print(f"Forward + Backward: {result_both.mean_ms:.2f} ms")
print(f"Backward overhead:  {result_both.mean_ms - result_fwd.mean_ms:.2f} ms "
      f"({(result_both.mean_ms / result_fwd.mean_ms - 1) * 100:.0f}% more)")

## Part 2: Nsight Systems Profiling

The nsys_profile.py script must be run from the **command line** with `nsys`.
Here we show the commands you would run:

```bash
# Forward pass profiling
nsys profile -t cuda,nvtx -o profile_forward --force-overwrite \
    python nsys_profile.py --mode forward --num_layers 6 --hidden_dim 1024

# Forward + backward profiling
nsys profile -t cuda,nvtx -o profile_fwd_bwd --force-overwrite \
    python nsys_profile.py --mode fwd_bwd

# Full training step (forward + backward + AdamW)
nsys profile -t cuda,nvtx -o profile_train --force-overwrite \
    python nsys_profile.py --mode train

# View kernel statistics
nsys stats profile_forward.nsys-rep
```

### Key questions to answer from nsys output:

1. **Q1**: Does the total forward pass time in nsys match the Python timeit measurement above?
2. **Q2**: Which CUDA kernel takes the most cumulative GPU time? Is it the same for fwd+bwd?
3. **Q3**: What non-matmul kernels account for non-trivial runtime?
4. **Q4**: How does the matmul time fraction change between inference and full training?
5. **Q5**: How does softmax runtime compare to matmul runtime in self-attention?

## Part 3: Memory Profiling

Profile peak GPU memory at different context lengths and modes.

In [ ]:
from memory_profiling import profile_peak_memory, calc_activation_size

# Clean up previous model
del model, input_ids, targets
torch.cuda.empty_cache()

In [ ]:
# (b) Peak memory across context lengths
seq_lens = [128, 256, 512]
modes = ["forward", "train"]

results_table = {}
for mode in modes:
    results_table[mode] = []
    for sl in seq_lens:
        peak = profile_peak_memory(
            num_layers=NUM_LAYERS,
            hidden_dim=HIDDEN_DIM,
            num_heads=NUM_HEADS,
            vocab_size=VOCAB_SIZE,
            batch_size=BATCH_SIZE,
            seq_len=sl,
            mode=mode,
        )
        results_table[mode].append(peak)
        print(f"  {mode:>8}, seq_len={sl:>4}: {peak:.1f} MB")

# Print nicely
print(f"\n{'Context Length':>16} | {'Forward (MB)':>14} | {'Train (MB)':>14}")
print(f"{'-'*16}-+-{'-'*14}-+-{'-'*14}")
for i, sl in enumerate(seq_lens):
    print(f"{sl:>16} | {results_table['forward'][i]:>14.1f} | {results_table['train'][i]:>14.1f}")

In [ ]:
# (c) Mixed-precision comparison
print("Mixed-precision (FP16) vs FP32 peak memory:\n")

for mode in ["forward", "train"]:
    peak_fp32 = profile_peak_memory(
        num_layers=NUM_LAYERS, hidden_dim=HIDDEN_DIM,
        num_heads=NUM_HEADS, vocab_size=VOCAB_SIZE,
        batch_size=BATCH_SIZE, seq_len=512,
        mode=mode, mixed_precision=False,
    )
    peak_fp16 = profile_peak_memory(
        num_layers=NUM_LAYERS, hidden_dim=HIDDEN_DIM,
        num_heads=NUM_HEADS, vocab_size=VOCAB_SIZE,
        batch_size=BATCH_SIZE, seq_len=512,
        mode=mode, mixed_precision=True,
    )
    reduction = (1 - peak_fp16 / peak_fp32) * 100
    print(f"  {mode:>8}: FP32={peak_fp32:.1f} MB, FP16={peak_fp16:.1f} MB "
          f"({reduction:.1f}% reduction)")

In [ ]:
# (d) Theoretical activation tensor size
calc_activation_size(
    batch_size=BATCH_SIZE,
    seq_len=512,
    hidden_dim=HIDDEN_DIM,
)

### Memory Snapshot Export

To generate memory timeline visualizations, run from the command line:

```bash
# Forward pass snapshot
python memory_profiling.py --mode forward --seq_len 512 --snapshot snapshot_fwd.pickle

# Full training step snapshot
python memory_profiling.py --mode train --seq_len 512 --snapshot snapshot_train.pickle
```

Then open the `.pickle` files at [pytorch.org/memory_viz](https://pytorch.org/memory_viz)
and select **"Active Memory Timeline"** to see how memory evolves during execution.

**What to look for:**
- Forward pass: memory climbs as activations accumulate across layers
- Backward pass: memory spikes then drops as gradients are computed and activations freed
- Optimizer step: brief spike for optimizer state (momentum, variance in AdamW)

## CLI Reference

All three scripts can also be run directly from the command line:

```bash
# Benchmarking
python benchmarking_script.py --mode forward --num_layers 6 --hidden_dim 1024
python benchmarking_script.py --mode both --warmup 10 --steps 50

# Nsight Systems
nsys profile -t cuda,nvtx -o profile_forward --force-overwrite \
    python nsys_profile.py --mode forward
nsys profile -t cuda,nvtx -o profile_train --force-overwrite \
    python nsys_profile.py --mode train
nsys stats profile_forward.nsys-rep

# Memory profiling
python memory_profiling.py --mode forward --seq_lens 128 256 512
python memory_profiling.py --mode train --seq_lens 128 256 512 --mixed_precision
python memory_profiling.py --mode forward --seq_len 512 --snapshot snapshot.pickle
python memory_profiling.py --calc_activation_size --batch_size 8 --seq_len 512 --hidden_dim 2560
```